# Regularization (Lasso, Ridge, ElasticNet)
**Repositori**: Machine Learning
**Topik**: Implementasi Regularization untuk mencegah overfitting
**Dataset**: Social_Network_Ads.csv
---
**Pendahuluan**: Regularization menambahkan penalty term pada fungsi loss untuk mencegah overfitting. Lasso (L1), Ridge (L2), dan ElasticNet (L1+L2) adalah teknik yang umum digunakan.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Load & EDA


In [ ]:
df = pd.read_csv('../../data/Social_Network_Ads.csv')
print('Shape:', df.shape)
print(df.head())
print(df['Purchased'].value_counts())
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(df[df['Purchased']==0]['Age'], bins=20, color='red', alpha=0.5, label='Not Purchased')
sns.histplot(df[df['Purchased']==1]['Age'], bins=20, color='blue', alpha=0.5, label='Purchased')
plt.title('Distribusi Age per Kelas')
plt.legend()
plt.subplot(1, 2, 2)
sns.histplot(df[df['Purchased']==0]['EstimatedSalary'], bins=20, color='red', alpha=0.5, label='Not Purchased')
sns.histplot(df[df['Purchased']==1]['EstimatedSalary'], bins=20, color='blue', alpha=0.5, label='Purchased')
plt.title('Distribusi EstimatedSalary per Kelas')
plt.legend()
plt.tight_layout()
plt.show()


## 3. Data Preparation


In [ ]:
df['Gender'] = (df['Gender'] == 'Male').astype(int)
X = df[['Gender', 'Age', 'EstimatedSalary']]
y = df['Purchased']
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_poly = poly.fit_transform(X)
print('Fitur asli:', X.shape[1], '-> Fitur polynomial:', X_poly.shape[1])
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Logistic Regression Tanpa Regularization


In [ ]:
lr = LogisticRegression(penalty='none', solver='lbfgs', max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
print(f'Accuracy (no reg): {accuracy_score(y_test, y_pred):.4f}')
print('Koefisien:', lr.coef_.ravel()[:5], '...')


## 5. Ridge Regularization (L2)


In [ ]:
ridge = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
print(f'Accuracy (Ridge): {accuracy_score(y_test, y_pred_ridge):.4f}')


## 6. Lasso Regularization (L1)


In [ ]:
lasso = LogisticRegression(penalty='l1', C=1.0, solver='liblinear', max_iter=1000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)
print(f'Accuracy (Lasso): {accuracy_score(y_test, y_pred_lasso):.4f}')
print(f'Fitur dengan koefisien nol (L1): {np.sum(lasso.coef_ == 0)} / {len(lasso.coef_.ravel())}')


## 7. ElasticNet Regularization (L1+L2)


In [ ]:
enet = LogisticRegression(penalty='elasticnet', C=1.0, l1_ratio=0.5, solver='saga', max_iter=1000)
enet.fit(X_train_scaled, y_train)
y_pred_enet = enet.predict(X_test_scaled)
print(f'Accuracy (ElasticNet): {accuracy_score(y_test, y_pred_enet):.4f}')


## 8. Tuning Regularization Strength (C)


In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_scores, lasso_scores = [], []
for C in C_values:
    ridge = LogisticRegression(penalty='l2', C=C, solver='lbfgs', max_iter=1000).fit(X_train_scaled, y_train)
    lasso = LogisticRegression(penalty='l1', C=C, solver='liblinear', max_iter=1000).fit(X_train_scaled, y_train)
    ridge_scores.append(accuracy_score(y_test, ridge.predict(X_test_scaled)))
    lasso_scores.append(accuracy_score(y_test, lasso.predict(X_test_scaled)))
plt.plot(C_values, ridge_scores, 'o-', label='Ridge (L2)')
plt.plot(C_values, lasso_scores, 's-', label='Lasso (L1)')
plt.xscale('log')
plt.xlabel('C (inverse regularization strength)')
plt.ylabel('Accuracy')
plt.title('Pengaruh Regularization Strength terhadap Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## 9. Perbandingan Koefisien


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model, name in zip(axes, [ridge, lasso, enet], ['Ridge (L2)', 'Lasso (L1)', 'ElasticNet']):
    ax.bar(range(len(model.coef_.ravel())), model.coef_.ravel())
    ax.set_title(f'Koefisien - {name}')
    ax.set_xlabel('Fitur')
    ax.set_ylabel('Koefisien')
plt.tight_layout()
plt.show()


## 10. Kesimpulan
Lasso (L1) menghasilkan fitur yang sparse (banyak koefisien nol) - berguna untuk feature selection. Ridge (L2) mendistribusikan bobot secara merata. ElasticNet menggabungkan kelebihan keduanya. Nilai C yang optimal ditemukan melalui tuning.
